In [ ]:
!pip install mp-api pandas numpy matminer pymatgen

In [ ]:
import os
import pandas as pd
import numpy as np
from mp_api.client import MPRester
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

### Data from Materials Project

In [ ]:
API_KEY = "jl33C95rdd7AunXvuoRDciMJDa4xOfNe"

with MPRester(API_KEY) as mpr:
    results = mpr.materials.summary.search(
        has_props=["magnetism"],
        fields=[
            "material_id",
            "formula_pretty",
            "elements",
            "nsites",
            "density",
            "formation_energy_per_atom",
            "band_gap",
            "total_magnetization",
            "total_magnetization_normalized_formula_units",
            "ordering",
            "symmetry"
        ],
        num_chunks=None
    )
    print(f"{len(results)} materials")

##Build DataFrame & Sample

In [ ]:
rows = []
for d in results:
    rows.append({
        "material_id": d.material_id,
        "formula": d.formula_pretty,
        "ordering": str(d.ordering),
        "band_gap": d.band_gap,
        "density": d.density,
        "nsites": d.nsites,
        "formation_energy_per_atom": d.formation_energy_per_atom,
        "total_magnetization": d.total_magnetization,
        "magnetization_per_fu": d.total_magnetization_normalized_formula_units,
        "n_elements": len(d.elements) if d.elements else None,
        "spacegroup": d.symmetry.symbol if d.symmetry else None,
        "crystal_system": d.symmetry.crystal_system.value if d.symmetry else None,
    })

df_full = pd.DataFrame(rows)
df_full = df_full[df_full['ordering'] != 'Unknown']

print("All available counts per ordering type:")
print(df_full["ordering"].value_counts())

In [ ]:
# Randomly sample ~5000 per ordering type (or all if fewer available)
sampled_groups = []
for otype in df_full["ordering"].unique():
    group = df_full[df_full["ordering"] == otype]
    n = min(5000, len(group))
    sampled_groups.append(group.sample(n=n, random_state=42))

df = pd.concat(sampled_groups).reset_index(drop=True)

print("Final sampled counts per ordering type:")
print(df["ordering"].value_counts())
print(f"\nTotal rows: {len(df)}")

##Generate Compositional Features with Matminer (Magpie Preset)

In [ ]:
# Convert formula strings to pymatgen Composition objects
df['composition'] = df['formula'].apply(
    lambda x: Composition(x) if pd.notnull(x) else None
)

# Drop rows where composition conversion failed
before = len(df)
df = df[df['composition'].notnull()].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with invalid formulas. Remaining: {len(df)}")

In [ ]:
# Initialize Magpie featurizer
ep = ElementProperty.from_preset('magpie')

# Featurize — ignore_errors=True skips compounds that fail without crashing
df_featurized = ep.featurize_dataframe(df, col_id='composition', ignore_errors=True)

# Drop the pymatgen Composition column
df_featurized = df_featurized.drop(columns=['composition'])

print(f"Original columns: {len(df.columns)}")
print(f"Total columns after featurization: {len(df_featurized.columns)}")
print(f"New Magpie features added: {len(df_featurized.columns) - len(df.columns) + 1}")

In [ ]:
ep = ElementProperty.from_preset('magpie')
df_featurized = ep.featurize_dataframe(df, col_id='composition', ignore_errors=True)

# Keep only the most relevant Magpie columns
magpie_cols_to_keep = [
    'MagpieData mean Number',
    'MagpieData mean AtomicWeight',
    'MagpieData mean MeltingT',
    'MagpieData mean NUnfilled',
    'MagpieData mean NdUnfilled',
    'MagpieData mean NfUnfilled',
    'MagpieData mean Electronegativity',
    'MagpieData range Electronegativity',
    'MagpieData mean CovalentRadius',
    'MagpieData range CovalentRadius',
    'MagpieData mean GSmagmom',
    'MagpieData mean SpaceGroupNumber',
]

base_cols = ['material_id', 'formula', 'ordering', 'crystal_system', 'band_gap',
             'density', 'nsites', 'formation_energy_per_atom',
             'total_magnetization', 'magnetization_per_fu',
             'n_elements', 'spacegroup']

df_featurized = df_featurized[base_cols + magpie_cols_to_keep]

##Missing Value Check

In [ ]:
missing = df_featurized.isnull().sum()
missing_pct = (missing / len(df_featurized) * 100).round(2)

missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_%', ascending=False)

print("Columns with missing values:")
print(missing_df if len(missing_df) > 0 else "None")

In [ ]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

df.drop(columns=['composition'], errors='ignore').to_csv('data/raw/magnetic_materials_raw.csv', index=False)
df_featurized.to_csv('data/processed/materials_featurized.csv', index=False)

print(f"\nFinal dataset shape: {df_featurized.shape}")
print("\nFirst 5 rows (base columns only):")
base_cols = ['material_id', 'formula', 'ordering', 'crystal_system', 'band_gap',
             'density', 'nsites', 'formation_energy_per_atom', 'magnetization_per_fu']
print(df_featurized[base_cols].head())